# 21 · Load: OEWS wages (BLS NY metro 2018–2025, NYS DOL NYC 2026) → `staging`

**Source cards:** `bls_oews_metro.md`, `nysdol_oews_regions.md` (§6 issues, §7 staging tables). Explored in: notebook 03.

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__oews_metro_occupations` | year × job, NY metro (not NYC), cross-industry | 6,281 |
| `stg_nysdol__occupation_wages` | job, NYC (5 boroughs), one vintage | 702 |

Values stay as published: symbols (`**`, `*`, `#`) become NULL with a count and a flag, never silently. The metro is labeled metro; mapping old codes to SOC 2018 happens in core.

Reading the eight OEWS workbooks takes 2–3 minutes.

In [1]:
import io
import sys
import zipfile
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "21-load-oews-wages.ipynb"
NY_MSA = "35620"
conn = connect()
summary = []

## 1. OEWS metro: read each year's `MSA_` workbook, keep the NY metro

Issues 1–3: column names change by year (2018 layout, case, `OCC_GROUP` → `O_GROUP`, `LOC QUOTIENT`); the 2022 zip also holds Excel lock files (`~$…`), so the member is picked by its `MSA_` prefix; the year comes from the zip name (`oesm25ma` → 2025).

In [2]:
FILES = raw_files("bls_oews_metro")
RENAME = {"OCC_GROUP": "O_GROUP", "AREA_NAME": "AREA_TITLE", "LOC_Q": "LOC_QUOTIENT"}
parts = []
for f, path in zip(FILES.file, FILES.path):
    year = 2000 + int(Path(f).stem[4:6])
    z = zipfile.ZipFile(path)
    msa = next(n for n in z.namelist() if Path(n).name.startswith("MSA_"))
    d = pd.read_excel(io.BytesIO(z.read(msa)), sheet_name=0, dtype=str, keep_default_na=False)
    d.columns = [c.strip().upper().replace(" ", "_") for c in d.columns]
    d = d.rename(columns=RENAME)
    ny = d[d.AREA.str.strip() == NY_MSA].copy()
    log(f"{year} {Path(msa).name}: keep AREA = {NY_MSA}", len(d), len(ny))
    ny.insert(0, "year", year)
    ny["source_file"] = f"{f}!{msa}"
    parts.append(ny)
o = pd.concat(parts, ignore_index=True)
o = o.apply(lambda s: s.str.strip() if s.dtype == object else s)
print(f"{len(o):,} NY metro rows, {o.year.nunique()} years")

NEED = ["AREA", "AREA_TITLE", "OCC_CODE", "OCC_TITLE", "O_GROUP", "TOT_EMP", "EMP_PRSE", "JOBS_1000", "LOC_QUOTIENT",
        "H_MEAN", "A_MEAN", "MEAN_PRSE", "H_PCT10", "H_PCT25", "H_MEDIAN", "H_PCT75", "H_PCT90",
        "A_PCT10", "A_PCT25", "A_MEDIAN", "A_PCT75", "A_PCT90", "ANNUAL", "HOURLY"]
missing = {y: [c for c in NEED if c not in g.columns or g[c].isna().all()] for y, g in o.groupby("year")}
print("Columns missing by year:", {y: m for y, m in missing.items() if m} or "none")
assert not any(missing.values())

  2018 MSA_M2018_dl.xlsx: keep AREA = 35620: 143,905 -> 773 rows (-143,132)


  2019 MSA_M2019_dl.xlsx: keep AREA = 35620: 142,410 -> 759 rows (-141,651)


  2020 MSA_M2020_dl.xlsx: keep AREA = 35620: 140,051 -> 756 rows (-139,295)


  2021 MSA_M2021_dl.xlsx: keep AREA = 35620: 147,188 -> 804 rows (-146,384)


  2022 MSA_M2022_dl.xlsx: keep AREA = 35620: 148,204 -> 800 rows (-147,404)


  2023 MSA_M2023_dl.xlsx: keep AREA = 35620: 149,591 -> 803 rows (-148,788)


  2024 MSA_M2024_dl.xlsx: keep AREA = 35620: 150,176 -> 794 rows (-149,382)


  2025 MSA_M2025_dl.xlsx: keep AREA = 35620: 150,023 -> 792 rows (-149,231)
6,281 NY metro rows, 8 years
Columns missing by year: none


## 2. Symbols in numeric columns → NULL, counted and flagged (issues 4–5)

`**` = employment not available, `*` = wage not available, `#` = wage at or above the top-code cap ($100/hr to 2021, $115/hr 2022–24). 2025 has no `#`: BLS stopped top-coding without notice (nb 03), so 2025 shows real top wages.

In [3]:
NUMS = {"TOT_EMP": "employment", "EMP_PRSE": "employment_rse_pct", "JOBS_1000": "jobs_per_1000",
        "LOC_QUOTIENT": "location_quotient", "H_MEAN": "hourly_mean", "A_MEAN": "annual_mean", "MEAN_PRSE": "mean_rse_pct",
        **{f"H_{p}": f"hourly_{p.lower().replace('pct', 'p').replace('median', 'p50')}" for p in ["PCT10", "PCT25", "MEDIAN", "PCT75", "PCT90"]},
        **{f"A_{p}": f"annual_{p.lower().replace('pct', 'p').replace('median', 'p50')}" for p in ["PCT10", "PCT25", "MEDIAN", "PCT75", "PCT90"]}}
WAGE = [c for c in NUMS if c[:2] in ("H_", "A_")]

o["is_top_coded"] = o[WAGE].eq("#").any(axis=1)
counts = []
for src, dst in NUMS.items():
    v = o[src].str.replace(",", "", regex=False)
    sym = v.isin(["*", "**", "#", ""])
    other = pd.to_numeric(v[~sym], errors="coerce").isna()
    assert not other.any(), f"{src}: unexpected text {v[~sym][other].unique()[:5]}"
    counts.append({"column": dst, **{s or "blank": (v == s).sum() for s in ["**", "*", "#", ""]}})
    o[dst] = pd.to_numeric(v.mask(sym), errors="raise").astype("float64")
display(pd.DataFrame(counts).set_index("column").T)
print("Top-coded rows by year:", o.groupby("year").is_top_coded.sum().to_dict())
assert o.loc[o.year == 2025, "is_top_coded"].sum() == 0
assert (o.employment.dropna() % 1 == 0).all()
o["employment"] = o.employment.astype("Int64")

column,employment,employment_rse_pct,jobs_per_1000,location_quotient,hourly_mean,annual_mean,mean_rse_pct,hourly_p10,hourly_p25,hourly_p50,hourly_p75,hourly_p90,annual_p10,annual_p25,annual_p50,annual_p75,annual_p90
**,191,191,191,191,0,0,0,0,0,0,0,0,0,0,0,0,0
*,0,0,0,0,514,100,68,514,514,514,514,514,100,100,100,100,100
#,0,0,0,0,1,1,0,1,10,72,200,378,1,10,78,214,427
blank,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Top-coded rows by year: {2018: 54, 2019: 60, 2020: 58, 2021: 77, 2022: 62, 2023: 57, 2024: 64, 2025: 0}


## 3. Codes and flags (issues 6, 9)

SOC edition by year: 2018 = SOC 2010, 2019–2020 = BLS hybrid, 2021+ = SOC 2018. `O_GROUP` → `occ_level`. `ANNUAL` / `HOURLY` = TRUE when only annual (teachers, pilots) or only hourly wages are published.

In [4]:
o["soc_edition"] = o.year.map(lambda y: "2010" if y == 2018 else "hybrid" if y <= 2020 else "2018")
print(o.O_GROUP.value_counts().to_dict())
o["occ_level"] = o.O_GROUP.str.lower().replace({"total": "total", "major": "major", "detailed": "detailed"})
assert set(o.occ_level) <= {"total", "major", "detailed"}, set(o.occ_level)
for c in ["ANNUAL", "HOURLY"]:
    assert set(o[c].str.upper()) <= {"TRUE", ""}, set(o[c])
o["annual_only"] = o.ANNUAL.str.upper().eq("TRUE")
o["hourly_only"] = o.HOURLY.str.upper().eq("TRUE")
print(o.groupby("year")[["annual_only", "hourly_only"]].sum().T)

# Tie-out (card §5): BLS release, NY metro May 2025, mean hourly wage all jobs $41.50
t = o[(o.year == 2025) & (o.OCC_CODE == "00-0000")].hourly_mean.iloc[0]
print(f"2025 all-jobs mean hourly: ${t:.2f} (BLS release: $41.50)"); assert round(t, 2) == 41.50

stg = (o.rename(columns={"AREA": "area_code", "AREA_TITLE": "area_title", "OCC_CODE": "occ_code", "OCC_TITLE": "occ_title"})
        [["year", "area_code", "area_title", "occ_code", "occ_title", "soc_edition", "occ_level", *NUMS.values(),
          "annual_only", "hourly_only", "is_top_coded", "source_file"]]
        .sort_values(["year", "occ_code"]).reset_index(drop=True))
stg["year"] = stg.year.astype("Int64")
summary.append(stage(stg, "stg_bls__oews_metro_occupations", ["year", "occ_code"], expected_rows=6_281,
                     source_key="bls_oews_metro", notebook=NB, conn=conn,
                     comment="BLS OEWS, New York-Newark-Jersey City metro (35620), NOT NYC. Codes: SOC 2010 (2018), hybrid (2019-20), SOC 2018 (2021+)."))

{'detailed': 6097, 'major': 176, 'total': 8}
year         2018  2019  2020  2021  2022  2023  2024  2025
annual_only    57    57    56    58    58    58    57    57
hourly_only     4     4     4     4     5     5     5     5
2025 all-jobs mean hourly: $41.50 (BLS release: $41.50)


  stg_bls__oews_metro_occupations: 6,281 rows x 28 cols, key ['year', 'occ_code'] unique | local data/interim/staging/stg_bls__oews_metro_occupations.parquet | DB: 6281


## 4. NYS DOL wages: NYC, one vintage (issues 10–14)

Padded column names; blank employment (suppressed) and blank experienced wage stay NULL, not zero; wages floored at the 2026 NYC minimum wage ($17/hr × 2,080 = $35,360) get a flag. Wages are aged to Q1 2026 from the May 2025 survey.

In [5]:
wf = raw_files("nysdol_oews_regions").iloc[-1]
w = pd.read_csv(wf.path, dtype=str, keep_default_na=False)
w.columns = [c.strip() for c in w.columns]
w = w.apply(lambda s: s.str.strip())
nyc = w[w["Area Name"] == "New York City"].copy()
log("keep Area Name = New York City", len(w), len(nyc))

WN = {"Employment": "employment", "Mean Wage": "mean_wage", "Median Wage": "median_wage",
      "Entry Wage": "entry_wage", "Experienced Wage": "experienced_wage"}
for src, dst in WN.items():
    blank = nyc[src] == ""
    print(f"  {dst:<17} blank -> NULL: {blank.sum()}")
    nyc[dst] = pd.to_numeric(nyc[src].mask(blank), errors="raise").astype("Int64")

FLOOR = 17 * 2080
print(f"Lowest entry wage: {nyc.entry_wage.min():,} (floor {FLOOR:,})"); assert nyc.entry_wage.min() == FLOOR
nyc["entry_wage_at_floor"] = nyc.entry_wage.eq(FLOOR).fillna(False).astype(bool)
nyc["median_at_floor"] = nyc.median_wage.eq(FLOOR).fillna(False).astype(bool)
print("At floor: entry", nyc.entry_wage_at_floor.sum(), "| median", nyc.median_at_floor.sum())

def soc_level(s):
    t = s[3:]
    return "total" if s == "00-0000" else "major" if t == "0000" else "minor" if t.endswith("00") else "broad" if t.endswith("0") else "detailed"

nyc["soc_level"] = nyc["Standard Occupational Code"].map(soc_level)
nyc["vintage"] = wf.downloaded_utc[:10]
nyc["wage_reference"] = "2026Q1"
nyc["survey_reference"] = "May 2025"
nyc["source_file"] = wf.file

# Tie-out (card §5): NYC total 4,603,380 jobs, mean $95,518
tot = nyc[nyc["Standard Occupational Code"] == "00-0000"].iloc[0]
print(f"NYC total: {tot.employment:,} jobs, mean ${tot.mean_wage:,}"); assert (tot.employment, tot.mean_wage) == (4_603_380, 95_518)

stg_w = (nyc.rename(columns={"Area": "area_code", "Area Name": "area_name", "Standard Occupational Code": "soc_code",
                             "Occupational Title": "soc_title"})
          [["vintage", "wage_reference", "survey_reference", "area_code", "area_name", "soc_code", "soc_title", "soc_level",
            *WN.values(), "entry_wage_at_floor", "median_at_floor", "source_file"]]
          .sort_values("soc_code").reset_index(drop=True))
summary.append(stage(stg_w, "stg_nysdol__occupation_wages", ["area_code", "soc_code"], expected_rows=702,
                     source_key="nysdol_oews_regions", notebook=NB, conn=conn,
                     comment="NYS DOL OEWS wages for NYC (5 boroughs), annual $, aged to 2026Q1; entry = mean of bottom third."))

  keep Area Name = New York City: 7,036 -> 702 rows (-6,334)
  employment        blank -> NULL: 117
  mean_wage         blank -> NULL: 0
  median_wage       blank -> NULL: 0
  entry_wage        blank -> NULL: 0
  experienced_wage  blank -> NULL: 6
Lowest entry wage: 35,360 (floor 35,360)
At floor: entry 22 | median 5
NYC total: 4,603,380 jobs, mean $95,518
  stg_nysdol__occupation_wages: 702 rows x 16 cols, key ['area_code', 'soc_code'] unique | local data/interim/staging/stg_nysdol__occupation_wages.parquet | DB: 702


## 5. Summary

In [6]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__oews_metro_occupations,6281,28,year + occ_code,bls_oews_metro,21-load-oews-wages.ipynb,6281,2026-10-14T21:12:04+00:00
1,stg_nysdol__occupation_wages,702,16,area_code + soc_code,nysdol_oews_regions,21-load-oews-wages.ipynb,702,2026-10-14T21:12:04+00:00
